# 🐐 AlpasFarm: Train YOLOv8 on 4skwhnrscr Goat Anatomical & Health Dataset

This official Google Colab notebook automates end-to-end training of **YOLOv8** on the **4skwhnrscr-2** goat dataset (2,991 images, 15,072 annotations across `goat_face`, `eye`, `mouth`, `ear`, and `goat_body`).

**Detected Anatomical & Clinical Targets:**
- **Class 0 (`goat_face`)**: Facial landmark detection & symmetry
- **Class 1 (`eye`)**: Ocular screening (FAMACHA conjunctival pallor, discharge, opacity)
- **Class 2 (`mouth`)**: Muzzle inspection (Contagious Ecthyma / Orf scabs, nasal discharge)
- **Class 3 (`ear`)**: Ear posture (drooping lethargy indicator, mange mite lesions)
- **Class 4 (`goat_body`)**: Full body posture, BCS (1-5), and bloat detection

### Step 1: Install Dependencies & Check GPU Acceleration

In [ ]:
!pip install -q ultralytics torchvision pillow matplotlib pyyaml
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

### Step 2: Upload or Download 4skwhnrscr-2 Dataset
Run this cell to extract the dataset archives or download them directly.

In [ ]:
import os, tarfile, shutil
from pathlib import Path

DATA_DIR = Path("dataset_4sk")
DATA_DIR.mkdir(exist_ok=True)

# If uploading 4skwhnrscr-2 archive:
if Path("dataset_1.tar.xz").exists():
    print("Extracting dataset_1.tar.xz...")
    with tarfile.open("dataset_1.tar.xz", "r:*") as tar:
        tar.extractall(DATA_DIR / "raw_1")

if Path("dataset_2.tar.xz").exists():
    print("Extracting dataset_2.tar.xz...")
    with tarfile.open("dataset_2.tar.xz", "r:*") as tar:
        tar.extractall(DATA_DIR / "raw_2")

print("Dataset extraction complete!")

### Step 3: Create YAML Dataset Descriptor

In [ ]:
data_yaml = '''# AlpasFarm 4skwhnrscr-2 Dataset
path: /content/dataset_4sk/yolo_dataset
train: images/train
val: images/val
test: images/test

nc: 5
names: ['goat_face', 'eye', 'mouth', 'ear', 'goat_body']
'''

with open('data.yaml', 'w') as f:
    f.write(data_yaml)
print("data.yaml created successfully!")

### Step 4: Train YOLOv8 on GPU

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 model
model = YOLO('yolov8n.pt')

# Train for 50 epochs
results = model.train(
    data='data.yaml',
    epochs=50,
    batch=16,
    imgsz=640,
    device=0 if torch.cuda.is_available() else 'cpu',
    patience=15,
    save=True,
    plots=True,
    project='alpasfarm_runs',
    name='4sk_goat_model',
    exist_ok=True
)

### Step 5: Evaluate Model Performance & Confusion Matrix

In [ ]:
from IPython.display import Image, display

# Validate model
metrics = model.val()
print(f"mAP@0.5:      {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision:    {metrics.box.mp:.4f}")
print(f"Recall:       {metrics.box.mr:.4f}")

# Display training results and curves
results_png = Path('alpasfarm_runs/4sk_goat_model/results.png')
if results_png.exists():
    display(Image(filename=str(results_png)))

### Step 6: Export Trained Weights to ONNX & PyTorch for AlpasFarm

In [ ]:
# Export best model to ONNX
onnx_path = model.export(format='onnx', imgsz=640)
print(f"ONNX Model exported: {onnx_path}")

from google.colab import files
# Download best weights directly
files.download('alpasfarm_runs/4sk_goat_model/weights/best.pt')
if Path(onnx_path).exists():
    files.download(onnx_path)